In [1]:
!pip install -q -U transformers sentencepiece jiwer soundfile "datasets<4.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 38.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is

In [2]:
import glob, os, re, time
import numpy as np
import pandas as pd
import torch
torch.manual_seed(42)  # fix randomness so audio is reproducible
paths = glob.glob("/kaggle/input/**/summaries_for_audio.csv", recursive=True)
print("Found:", paths)
summ = pd.read_csv(paths[0])
summ["summary"] = summ["summary"].fillna("").astype(str)
print(summ.shape)
summ.head()

Found: ['/kaggle/input/datasets/sawsannoli/summaries-for-audio/summaries_for_audio.csv']
(250, 4)


,image,true_grade,pred_grade,summary
0,4095_right,0,0.0,The fundus image shows no signs of diabetic re...
1,11527_left,0,2.0,The image shows microaneurysms and hemorrhages...
2,35004_right,0,1.0,"The image shows microaneurysms, indicating mil..."
3,25941_left,0,1.0,"The image shows microaneurysms, indicating mil..."
4,28774_right,0,2.0,The image shows moderate non-proliferative dia...


In [3]:
ABBREV = {
    r"\bNPDR\b": "non proliferative diabetic retinopathy",
    r"\bPDR\b": "proliferative diabetic retinopathy",
    r"\bDR\b": "diabetic retinopathy",
    r"\bIRMA\b": "intraretinal microvascular abnormalities",
    r"\bOCT\b": "optical coherence tomography",
}

def normalize_for_tts(text):
    """Expand abbreviations and clean symbols so the TTS model can pronounce them."""
    for pat, rep in ABBREV.items():
        text = re.sub(pat, rep, text)
    text = text.replace("-", " ").replace("/", " or ")
    return re.sub(r"\s+", " ", text).strip()

summ["tts_text"] = summ["summary"].apply(normalize_for_tts)
print(summ["tts_text"].iloc[0])

The fundus image shows no signs of diabetic retinopathy. The patient should be monitored regularly to detect any changes.


In [4]:
import torch
from datasets import load_dataset
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan

device = "cuda" if torch.cuda.is_available() else "cpu"
tts_processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
tts_model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to(device)
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)

# Fixed speaker voice (x-vector) so every file uses the same voice
xvectors = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker = torch.tensor(xvectors[7306]["xvector"]).unsqueeze(0).to(device)
SR = 16000  # SpeechT5 output sampling rate
print("Loaded on", device)

preprocessor_config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.06k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/232 [00:00<?, ?B/s]

spm_char.model: reconstructing file:   0%|          |  0.00B /  238kB            

spm_char.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

default/validation/0000.parquet: reconstructing file:   0%|          |  0.00B / 21.3MB            

default/validation/0000.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/7931 [00:00<?, ? examples/s]

Loaded on cuda


In [5]:
import soundfile as sf
from IPython.display import Audio

def synthesize(text):
    """Generate speech sentence by sentence (SpeechT5 struggles with long inputs) and join with short pauses."""
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    pause = np.zeros(int(0.3 * SR), dtype=np.float32)
    parts = []
    for s in sentences:
        inputs = tts_processor(text=s, return_tensors="pt").to(device)
        with torch.inference_mode():
            wav = tts_model.generate_speech(inputs["input_ids"], speaker, vocoder=vocoder)
        parts += [wav.cpu().numpy(), pause]
    return np.concatenate(parts) if parts else np.zeros(SR, dtype=np.float32)

test = synthesize(summ["tts_text"].iloc[0])
print("Duration:", round(len(test) / SR, 1), "s")
Audio(test, rate=SR)

Duration: 9.3 s


In [6]:
AUDIO_DIR = "/kaggle/working/audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

records = []
for i, row in summ.iterrows():
    path = f"{AUDIO_DIR}/{row['image']}.wav"
    t0 = time.time()
    wav = synthesize(row["tts_text"])
    gen_time = time.time() - t0
    sf.write(path, wav, SR)
    dur = len(wav) / SR
    records.append({"image": row["image"], "audio_path": path, "duration_s": round(dur, 2),
                    "gen_time_s": round(gen_time, 2), "rtf": round(gen_time / dur, 3)})
    if (i + 1) % 25 == 0:
        print(f"{i + 1}/{len(summ)}")

audio_df = summ.merge(pd.DataFrame(records), on="image")
audio_df.to_csv("/kaggle/working/audio_index.csv", index=False)
print("Done. Avg duration:", audio_df["duration_s"].mean().round(1), "s")

25/250
50/250
75/250
100/250
125/250
150/250
175/250
200/250
225/250
250/250
Done. Avg duration: 13.0 s


In [7]:
import jiwer
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model="openai/whisper-small",
               device=0 if device == "cuda" else -1)

def clean(t):
    """Lowercase and drop punctuation so WER compares words only."""
    t = re.sub(r"[^a-z0-9 ]", " ", str(t).lower())
    return re.sub(r"\s+", " ", t).strip()

transcripts = []
for p in audio_df["audio_path"]:
    wav, sr = sf.read(p)
    out = asr({"raw": wav.astype(np.float32), "sampling_rate": sr},
              generate_kwargs={"language": "english"})
    transcripts.append(out["text"])

audio_df["transcript"] = transcripts
audio_df["wer"] = [jiwer.wer(clean(r), clean(h)) for r, h in zip(audio_df["tts_text"], audio_df["transcript"])]
print("Mean WER:", round(audio_df["wer"].mean(), 3), "| Median WER:", round(audio_df["wer"].median(), 3))

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Mean WER: 0.046 | Median WER: 0.042


In [8]:
# UTMOS predicts a 1-5 naturalness score; skipped gracefully if it cannot load
try:
    mos_model = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).to(device)
    scores = []
    for p in audio_df["audio_path"]:
        wav, sr = sf.read(p)
        w = torch.from_numpy(wav.astype(np.float32)).unsqueeze(0).to(device)
        with torch.inference_mode():
            scores.append(float(mos_model(w, sr)))
    audio_df["utmos"] = scores
    print("Mean UTMOS:", np.mean(scores).round(2))
except Exception as e:
    print("UTMOS skipped:", e)

Downloading: "https://github.com/tarepan/SpeechMOS/zipball/v1.2.0" to /root/.cache/torch/hub/v1.2.0.zip


/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Downloading: "https://github.com/tarepan/SpeechMOS/releases/download/v1.0.0/utmos22_strong_step7459_v1.pt" to /root/.cache/torch/hub/checkpoints/utmos22_strong_step7459_v1.pt


100%|██████████| 392M/392M [00:03<00:00, 112MB/s] 


Mean UTMOS: 4.23


In [9]:
import shutil

metrics = {
    "files": len(audio_df),
    "mean_duration_s": audio_df["duration_s"].mean(),
    "mean_rtf": audio_df["rtf"].mean(),
    "mean_wer": audio_df["wer"].mean(),
    "median_wer": audio_df["wer"].median(),
    "share_wer_0": (audio_df["wer"] == 0).mean(),
}
if "utmos" in audio_df:
    metrics["mean_utmos"] = audio_df["utmos"].mean()
pd.Series(metrics).round(3).to_csv("/kaggle/working/audio_metrics.csv")
print(pd.Series(metrics).round(3))

# WER by true grade (do more severe summaries have harder medical words?)
print(audio_df.groupby("true_grade")["wer"].mean().round(3))
audio_df.to_csv("/kaggle/working/audio_results.csv", index=False)

# 10 clips (2 per grade) for human listeners to rate 1-5
pick = audio_df.groupby("true_grade").sample(n=2, random_state=42)
os.makedirs("/kaggle/working/human_audio", exist_ok=True)
for p in pick["audio_path"]:
    shutil.copy(p, "/kaggle/working/human_audio/")
sheet = pick[["image", "tts_text"]].copy()
for col in ["clarity_1to5", "naturalness_1to5", "matches_text_1to5"]:
    sheet[col] = ""
sheet.to_csv("/kaggle/working/human_audio/rating_sheet.csv", index=False)

shutil.make_archive("/kaggle/working/audio_all", "zip", AUDIO_DIR)
print("Saved:", os.listdir("/kaggle/working"))

files              250.000
mean_duration_s     13.002
mean_rtf             0.188
mean_wer             0.046
median_wer           0.042
share_wer_0          0.256
mean_utmos           4.227
dtype: float64
true_grade
0    0.052
1    0.055
2    0.057
3    0.035
4    0.032
Name: wer, dtype: float64
Saved: ['audio_results.csv', 'audio_all.zip', '__notebook__.ipynb', 'audio_metrics.csv', 'human_audio', 'audio_index.csv', 'audio']
